<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/06_Interview_Prep/ai_engineer/01_llm_concepts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Engineer Interview — LLM Concepts with Code

The LLM fundamentals every AI Engineer is expected to know — each concept demonstrated with runnable code so you understand the *why*, not just the *what*.

**Topics:** Tokenization, embeddings, attention mechanics, sampling strategies, prompt engineering patterns, evaluation

## 1. Tokenization — What LLMs Actually See

In [ ]:
# pip install tiktoken
import tiktoken

# GPT-4 uses cl100k_base tokenizer (100k vocab, byte-pair encoding)
enc = tiktoken.get_encoding('cl100k_base')

def show_tokenization(text: str, encoding=enc) -> None:
    tokens = encoding.encode(text)
    decoded = [encoding.decode_single_token_bytes(t).decode('utf-8', errors='replace') for t in tokens]
    print(f'Text:   {repr(text)}')
    print(f'Tokens: {tokens}')
    print(f'Pieces: {decoded}')
    print(f'Count:  {len(tokens)}')
    print()

# Critical for cost estimation and context window management
examples = [
    'Hello world',
    'ChatGPT is amazing',
    'Tokenization is surprisingly non-obvious.',
    'antiestablishmentarianism',
    '   spaces   matter   a lot   ',
    '1+1=2 and 100+200=300',
    '\n\n\n',  # whitespace tokens
]

for text in examples:
    show_tokenization(text)

# Cost estimation
print('Token cost estimation (GPT-4o pricing):')
input_cost_per_1k = 0.005   # $5 / 1M tokens input
output_cost_per_1k = 0.015  # $15 / 1M tokens output

def estimate_cost(prompt: str, expected_output_tokens: int) -> dict:
    input_tokens = len(enc.encode(prompt))
    cost = (input_tokens / 1000 * input_cost_per_1k +
            expected_output_tokens / 1000 * output_cost_per_1k)
    return {'input_tokens': input_tokens, 'output_tokens': expected_output_tokens,
            'total_tokens': input_tokens + expected_output_tokens, 'cost_usd': cost}

short_prompt = 'What is 2+2?'
long_prompt = ' '.join(['word'] * 1000)  # 1000-word prompt

for name, prompt in [('Short prompt', short_prompt), ('Long prompt (1000w)', long_prompt)]:
    est = estimate_cost(prompt, expected_output_tokens=200)
    print(f'  {name}: {est["input_tokens"]} in + {est["output_tokens"]} out = ${est["cost_usd"]:.5f}')

print()
print('Interview insight: Context window = max tokens (input + output combined).')
print('GPT-4o: 128k tokens ≈ 95k words ≈ 300 pages.')
print('Chunking strategy matters: 1 token ≈ 0.75 words (English), fewer for code/math.')

## 2. Embeddings — Semantic Meaning as Vectors

In [ ]:
import numpy as np

# Demonstrate embedding properties without API calls
# Using pre-defined vectors to show key concepts

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    return np.linalg.norm(a - b)

def dot_product_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b)

# Simulate word embeddings (simplified 3D for illustration)
# Real embeddings: 1536 dims (OpenAI text-embedding-3-small)
embeddings = {
    'king':    np.array([0.9, 0.1, 0.8]),   # high royalty, low female, high prestige
    'queen':   np.array([0.9, 0.9, 0.8]),   # high royalty, high female, high prestige
    'man':     np.array([0.1, 0.1, 0.5]),
    'woman':   np.array([0.1, 0.9, 0.5]),
    'dog':     np.array([0.0, 0.2, 0.1]),
    'puppy':   np.array([0.0, 0.3, 0.05]),  # similar to dog
    'car':     np.array([0.0, 0.0, 0.9]),
}

# Classic: king - man + woman ≈ queen
analogy = embeddings['king'] - embeddings['man'] + embeddings['woman']
print('Word Analogy: king - man + woman = ?')
for word, emb in embeddings.items():
    sim = cosine_similarity(analogy, emb)
    print(f'  {word:<10}: cosine similarity = {sim:.4f}')

print()
print('Similarity metrics comparison:')
pairs = [('king', 'queen', 'Related (royalty)'), ('dog', 'puppy', 'Very similar'), ('king', 'car', 'Unrelated')]
print(f'{"Pair":<20} {"Cosine":>10} {"Euclidean":>12} {"Dot Product":>13}')
for w1, w2, label in pairs:
    e1, e2 = embeddings[w1], embeddings[w2]
    print(f'{label:<20} {cosine_similarity(e1,e2):>10.4f} {euclidean_distance(e1,e2):>12.4f} {dot_product_similarity(e1,e2):>13.4f}')

print()
print('When to use each similarity metric:')
print('  Cosine:     Best for semantic similarity (direction matters, not magnitude)')
print('  Euclidean:  When magnitude matters; sensitive to outliers')
print('  Dot product: Used inside attention (before softmax); magnitude matters')
print()
print('Interview Q: Why do vector databases like FAISS default to cosine/inner product?')
print('  → Normalized vectors: cosine sim = dot product, enabling fast BLAS operations')
print()

# Dimensionality and model selection
embedding_models = [
    ('text-embedding-3-small', 1536, 0.00002, 'Best cost/quality for most tasks'),
    ('text-embedding-3-large', 3072, 0.00013, 'Highest quality, 2× dimensions'),
    ('text-embedding-ada-002', 1536, 0.0001,  'Legacy, replaced by 3-small'),
    ('nomic-embed-text',        768, 0.00000, 'Free/open-source alternative'),
    ('bge-m3',                 1024, 0.00000, 'Multilingual, strong benchmarks'),
]
print('Embedding model comparison:')
print(f'{"Model":<30} {"Dims":>6} {"$/1k tokens":>12} {"Notes"}')
for model, dims, cost, notes in embedding_models:
    print(f'{model:<30} {dims:>6} {cost:>12.5f} {notes}')

## 3. Sampling Strategies — Controlling Output Quality

In [ ]:
import numpy as np

# Simulate token probability distributions and sampling strategies

def softmax(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    """Apply temperature scaling then softmax."""
    scaled = logits / temperature
    exp = np.exp(scaled - scaled.max())
    return exp / exp.sum()

def top_k_sample(probs: np.ndarray, k: int) -> np.ndarray:
    """Zero out all but top-k probabilities, then renormalize."""
    top_k_idx = np.argsort(probs)[::-1][:k]
    filtered = np.zeros_like(probs)
    filtered[top_k_idx] = probs[top_k_idx]
    return filtered / filtered.sum()

def top_p_sample(probs: np.ndarray, p: float) -> np.ndarray:
    """Nucleus sampling: keep smallest set of tokens with cumulative prob >= p."""
    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    cumsum = np.cumsum(sorted_probs)
    # Keep tokens until cumulative prob exceeds p
    cutoff = np.searchsorted(cumsum, p) + 1
    filtered = np.zeros_like(probs)
    filtered[sorted_idx[:cutoff]] = probs[sorted_idx[:cutoff]]
    return filtered / filtered.sum()

# Simulated vocabulary logits (next token after "The weather today is")
vocab = ['sunny', 'cloudy', 'rainy', 'perfect', 'hot', 'cold', 'terrible',
          'gorgeous', 'ok', 'moderate', 'the', 'a', 'very', 'quite', 'somewhat']
# Assume these logits from a model
logits = np.array([3.5, 2.8, 2.1, 2.9, 1.8, 1.5, 0.5, 2.7, 0.8, 0.7,
                    -1.0, -1.5, 0.2, 0.1, -0.2])

print('Sampling strategies for next-token prediction:')
print('Context: "The weather today is __"')
print()

strategies = [
    ('Greedy (temp=0)',     softmax(logits, 0.01)),
    ('Temp=0.3',            softmax(logits, 0.3)),
    ('Temp=1.0',            softmax(logits, 1.0)),
    ('Temp=1.5',            softmax(logits, 1.5)),
    ('Top-k (k=3)',         top_k_sample(softmax(logits, 1.0), k=3)),
    ('Top-p (p=0.9)',       top_p_sample(softmax(logits, 1.0), p=0.9)),
]

# Show top-5 tokens for each strategy
for strat_name, probs in strategies:
    top5_idx = np.argsort(probs)[::-1][:5]
    top5 = [(vocab[i], probs[i]) for i in top5_idx if probs[i] > 0]
    entropy = -np.sum(probs * np.log(probs + 1e-10))
    print(f'{strat_name:<20} entropy={entropy:.2f}  top tokens: {", ".join(f"{w}({p:.3f})" for w,p in top5[:5])}')

print()
print('Interview quick-reference:')
tips = [
    ('temperature=0', 'Deterministic greedy decoding'),
    ('temperature<1', 'More conservative, less diverse'),
    ('temperature>1', 'More random, may become incoherent'),
    ('top_k=50', 'Always sample from top 50 tokens'),
    ('top_p=0.9', 'Sample from tokens covering 90% probability mass'),
    ('top_p=1.0', 'All tokens eligible (effectively disabled)'),
]
for param, meaning in tips:
    print(f'  {param:<18}: {meaning}')
print()
print('Best practice: Use top_p=0.9 + temperature=0.7 for most generation tasks.')
print('For structured output (JSON, code): temperature=0, or use json_mode.')

## 4. RAG Debugging Framework — Systematic Diagnosis

In [ ]:
import numpy as np
from dataclasses import dataclass
from enum import Enum

class RAGFailureMode(Enum):
    RETRIEVAL_RECALL = 'retrieval_recall'    # right doc not in top-k
    RETRIEVAL_PRECISION = 'retrieval_precision'  # wrong docs in top-k
    GENERATION = 'generation'               # correct context, wrong answer
    CHUNKING = 'chunking'                   # context split at wrong boundary
    QUERY = 'query'                         # query doesn't match doc language

@dataclass
class RAGDiagnostic:
    question: str
    expected_answer: str
    retrieved_chunks: list[str]
    generated_answer: str
    ground_truth_in_retrieved: bool

def diagnose_rag_failure(diag: RAGDiagnostic) -> tuple[RAGFailureMode, str]:
    """Identify the failure mode in a RAG pipeline."""
    # Decision tree for failure diagnosis
    if not diag.ground_truth_in_retrieved:
        # The answer wasn't even in the retrieved chunks → retrieval failure
        if not diag.retrieved_chunks:
            return RAGFailureMode.RETRIEVAL_RECALL, 'No chunks retrieved at all — check embedding or index'
        # Check if any retrieved chunk is even on-topic
        keyword_overlap = any(
            any(word.lower() in chunk.lower() for word in diag.question.split())
            for chunk in diag.retrieved_chunks
        )
        if not keyword_overlap:
            return RAGFailureMode.QUERY, 'Query vocabulary doesn\'t match document language → try HyDE or query expansion'
        return RAGFailureMode.RETRIEVAL_RECALL, 'Right doc not retrieved — increase k or improve embedding model'

    # Ground truth WAS retrieved but answer is still wrong
    if diag.expected_answer.lower() not in diag.generated_answer.lower():
        # Is the answer a substring of a retrieved chunk?
        answer_in_chunk = any(diag.expected_answer.lower() in chunk.lower()
                               for chunk in diag.retrieved_chunks)
        if not answer_in_chunk:
            return RAGFailureMode.CHUNKING, 'Answer spans chunk boundary — reduce chunk overlap or increase chunk size'
        return RAGFailureMode.GENERATION, 'Context retrieved correctly but model ignored it — check prompt, try chain-of-thought'

    return RAGFailureMode.RETRIEVAL_PRECISION, 'Answer found but answer quality is poor — add reranking'

# Test cases
test_cases = [
    RAGDiagnostic(
        question='What year was Python created?',
        expected_answer='1991',
        retrieved_chunks=['JavaScript was created in 1995 by Brendan Eich.',
                           'Ruby was released in 1995.'],
        generated_answer="I don't know based on the available context.",
        ground_truth_in_retrieved=False
    ),
    RAGDiagnostic(
        question='What is the capital of France?',
        expected_answer='Paris',
        retrieved_chunks=['France is a country in Western Europe. Paris, the capital city, has...'],
        generated_answer='The capital of Spain is Madrid.',  # model hallucinated
        ground_truth_in_retrieved=True
    ),
    RAGDiagnostic(
        question='GPT-4 context window size',
        expected_answer='128k tokens',
        retrieved_chunks=['The model supports up to 128', 'k tokens in its context window.'],  # split!
        generated_answer='The context window size is not mentioned.',
        ground_truth_in_retrieved=False
    ),
]

print('RAG Failure Diagnosis:')
print('-' * 70)
for i, case in enumerate(test_cases):
    failure_mode, recommendation = diagnose_rag_failure(case)
    print(f'Case {i+1}: "{case.question}"')
    print(f'  Failure mode:   {failure_mode.value}')
    print(f'  Recommendation: {recommendation}')
    print()

print('RAG evaluation metrics to track:')
metrics = [
    ('Context Recall@k', 'Is the ground truth in top-k retrieved chunks?', '> 90%'),
    ('Context Precision', 'Of retrieved chunks, fraction that are actually useful', '> 70%'),
    ('Faithfulness', 'Is the answer grounded in (not contradicting) retrieved context?', '> 85%'),
    ('Answer Relevance', 'Does the answer actually address the question?', '> 85%'),
]
for metric, definition, target in metrics:
    print(f'  {metric:<22}: {definition:<60} Target: {target}')

## 5. Prompt Engineering Patterns — Production-Ready Templates

In [ ]:
import json
import re
from typing import Any

# ── Pattern 1: Structured Output with Validation ────────────────────────

EXTRACTION_SYSTEM = """You are a data extraction assistant.
Extract the requested information from the user's text and return ONLY valid JSON.
If a field cannot be found, use null."""

EXTRACTION_USER_TEMPLATE = """Extract the following fields from the text below:
- name (string): full name of the person
- email (string): email address
- company (string): company name
- role (string): job title

TEXT:
{text}

Return JSON only. No explanation."""

def parse_llm_json(response: str, schema: dict) -> dict:
    """Robustly parse JSON from LLM output, with fallback extraction."""
    # Try direct parse
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        pass
    # Try extracting JSON block from markdown
    match = re.search(r'```(?:json)?\s*({.*?})\s*```', response, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    # Try finding first { ... } block
    match = re.search(r'{.*}', response, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    # Return null for all fields
    return {k: None for k in schema.keys()}

print('Structured Output Pattern — JSON Extraction')
print('='*50)

# Simulated LLM responses (as if from API call)
simulated_responses = [
    '{"name": "John Smith", "email": "john@techcorp.com", "company": "TechCorp", "role": "CTO"}',
    '```json\n{"name": "Jane Doe", "email": null, "company": "StartupXYZ", "role": "ML Engineer"}\n```',
    'Here is the extracted info: {"name": "Bob", "email": "bob@example.com", "company": null, "role": "Data Scientist"}',
    'I could not extract the information requested.',  # failure case
]

schema = {'name': None, 'email': None, 'company': None, 'role': None}
for resp in simulated_responses:
    result = parse_llm_json(resp, schema)
    print(f'  Input:  {resp[:60]}...' if len(resp) > 60 else f'  Input:  {resp}')
    print(f'  Parsed: {result}')
    print()

# ── Pattern 2: Chain-of-Thought for Reasoning Tasks ─────────────────────

COT_TEMPLATE = """Solve the following problem step by step.
Show your reasoning clearly, then provide your final answer.

Problem: {problem}

Let me think step by step:"""

# Few-shot CoT examples
FEW_SHOT_COT = [
    {
        'problem': 'A store has 120 apples. It sells 35 in the morning and 28 in the afternoon. How many are left?',
        'reasoning': '1. Start: 120 apples\n2. Morning sales: 120 - 35 = 85 apples\n3. Afternoon sales: 85 - 28 = 57 apples',
        'answer': '57 apples'
    },
    {
        'problem': 'If 3 ML engineers can train a model in 12 days, how many engineers are needed to train it in 4 days?',
        'reasoning': '1. Total work = 3 engineers × 12 days = 36 engineer-days\n2. Need to complete in 4 days: 36 / 4 = 9 engineers',
        'answer': '9 engineers'
    }
]

def build_few_shot_cot_prompt(examples: list[dict], question: str) -> str:
    """Build a few-shot CoT prompt from examples."""
    prompt_parts = []
    for ex in examples:
        prompt_parts.append(f"Problem: {ex['problem']}")
        prompt_parts.append(f"Let me think step by step:\n{ex['reasoning']}")
        prompt_parts.append(f"Answer: {ex['answer']}")
        prompt_parts.append('')
    prompt_parts.append(f'Problem: {question}')
    prompt_parts.append('Let me think step by step:')
    return '\n'.join(prompt_parts)

new_question = 'A dataset has 10,000 samples. 80% for training, 10% validation, 10% test. How many samples in each split?'
prompt = build_few_shot_cot_prompt(FEW_SHOT_COT, new_question)
print('Few-Shot Chain-of-Thought Prompt:')
print('='*50)
print(prompt)
print()
print('Why CoT works: Forces the model to "show its work" → more accurate on multi-step reasoning.')
print('Research (Wei et al., 2022): CoT prompting improves complex reasoning by 40-60%.')

## Exercises

1. **Token counting and cost optimizer:** Write a function that, given a list of documents and a context window size, finds the optimal chunking strategy (size + overlap) to maximize the number of documents that fit while minimizing cost. Test with chunks of 256, 512, 1024, 2048 tokens.

2. **Embedding-based FAQ system:** Given a list of FAQs and a user question, use cosine similarity on pre-computed embeddings to find the most relevant answer. Add a threshold — if max similarity < 0.7, respond "I don't know". Measure precision on a test set of 20 questions.

3. **Prompt A/B tester:** Implement a function that takes two prompt templates, runs each on the same 10 test inputs, and evaluates output quality using simple heuristics (length, keyword presence, JSON validity). Produce a comparison report.